# 01 - Data Profiling

Dataset: https://github.com/google/cluster-data/blob/master/ClusterData2019.md

### Purpose

This notebook performs **Data Profiling** for the Google ClusterData 2019 dataset. The goal is to understand the dataset structure, schema, and metadata. To determine the effecient strategy to do the data engineering and analysis in the next notebooks.

**Objectives:**
- Verify BigQuery connection to `google.com:google-cluster-data`
- Explore dataset metadata using **INFORMATION_SCHEMA** ($0 cost)
- Profile all 5 tables per cell: 
    - `machine_events`
    - `machine_attributes` 
    - `collection_events`
    - `instance_events` 
    - `instance_usage`
- Document table schemas, row counts, and usefulness rankings for the prediction analysis
- Maintain **$0 cost** by using dry runs and metadata queries only

## 1. Preparation
### 1.1. Import Library

In [14]:
import os
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery
from functions.fetch_cluster_data import get_bigquery_client

### 1.2. Configuration

In [15]:
# COST TRACKER
total_bytes_processed = 0
FREE_TIER_LIMIT_TIB = 1.0

def update_cost_tracker(bytes_processed):
    global total_bytes_processed
    total_bytes_processed += bytes_processed
    tib = total_bytes_processed / (1024**4)
    print(f"Estimated scan: {bytes_processed / 1e9:.3f} GB")
    print(f"Running total (estimated): {tib:.6f} TiB / {FREE_TIER_LIMIT_TIB:.1f} TiB free")
    return tib

print("✅ Cost tracker initialized. Goal: $0 (stay under 1 TiB/month)")
print(f"Free tier allowance: {FREE_TIER_LIMIT_TIB} TiB/month")

# Load environment variables and verify credentials
load_dotenv()

# Check required environment variables
gcp_project_id = os.environ.get('GCP_PROJECT_ID')
credentials_path = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')

print("=== Environment Check ===")
print(f"GCP_PROJECT_ID: {'✅ Set' if gcp_project_id else '❌ Not set'}")
print(f"GOOGLE_APPLICATION_CREDENTIALS: {'✅ Set' if credentials_path else '❌ Not set'}")

if credentials_path:
    print(f"Credentials path: {credentials_path}")
    print(f"File exists: {'✅ Yes' if os.path.exists(credentials_path) else '❌ No'}")

if not gcp_project_id:
    print("\n⚠️  Warning: GCP_PROJECT_ID not set in .env file")
    print("Create a .env file with: GCP_PROJECT_ID=your-project-id")

✅ Cost tracker initialized. Goal: $0 (stay under 1 TiB/month)
Free tier allowance: 1.0 TiB/month
=== Environment Check ===
GCP_PROJECT_ID: ✅ Set
GOOGLE_APPLICATION_CREDENTIALS: ✅ Set
Credentials path: sembada-cloud-research-key.json
File exists: ✅ Yes


### 1.3 Cost Control Strategy

**BigQuery Pricing (2026 Rates):**
- **$6.25 per TiB** scanned for on-demand queries
- **First 1 TiB/month is FREE** (our target: stay at $0)
- **INFORMATION_SCHEMA queries are FREE** (no bytes scanned)
- **Dry runs are FREE** (estimate bytes without executing)
Source: https://cloud.google.com/bigquery/pricing (Accessed at 2027-07-05)

**Cost Control Checklist [4]:**
1. ✅ Use `dry_run=True` before ANY query to verify 0 bytes
2. ✅ Use **INFORMATION_SCHEMA** heavily for metadata (FREE)
3. ✅ Use `LIMIT 5-10` for sample data (minimal scan)
4. ✅ Track `total_bytes_processed` (should show ~0 at end)
5. ✅ Never use `SELECT *` without LIMIT
6. ✅ Leverage 24-hour query caching (repeated queries FREE)

## 2. Connecting
### 2.1. Test Connection

In [16]:
# Initialize BigQuery client using helper function
try:
    client = get_bigquery_client(project_id=gcp_project_id)
    print("✅ BigQuery client initialized successfully!")
    print(f"Project: {client.project}")
    
    # Verify connection with a free metadata query
    query = """
    SELECT CURRENT_TIMESTAMP() AS current_time
    """
    # Note: @@project_name is MySQL-only, NOT available in BigQuery.
    # client.project already gives us the project ID.
    result = client.query(query).to_dataframe()
    print(f"\nConnection test: ✅ Success\n")
    print(result.to_string(index=False))
    
except ValueError as e:
    print(f"❌ Error: {e}")
    print("\nPlease ensure:")
    print("1. .env file exists with GCP_PROJECT_ID=your-project-id")
    print("2. credentials.json is valid and path is set in GOOGLE_APPLICATION_CREDENTIALS")
except Exception as e:
    print(f"❌ Unexpected error: {e}")


✅ BigQuery client initialized successfully!
Project: sembada-cloud-research-2026

Connection test: ✅ Success

                    current_time
2026-05-07 16:10:42.319158+00:00


### 2.2. Dry Test 

**Objectives:**

- This section verifies the BigQuery connection and explores the dataset structure using **FREE** metadata queries.

**Tasks:**
1. Test connection with a **DRY RUN** (estimated bytes = 0, cost = $0)
2. Query **INFORMATION_SCHEMA.TABLES** (FREE - metadata only)
3. List all 8 cells (a through h) and their tables
4. Verify the correct BigQuery path: `google.com:google-cluster-data`

**What is Dry Test?**

A dry run (or "dry test") in databases is a simulation of a query, script, or migration that allows you to see the results, potential conflicts, or SQL changes without actually modifying or committing data to the database. For this case we will try to estimate the bytes we will consume to just run one single query. Although we limit the data only 5 rows, BigQuery still reads all storage blocks for the selected columns because there’s no WHERE clause to prune partitions. LIMIT only reduces rows returned, not rows scanned. However, selecting fewer columns does reduce the amount of data read (columnar storage).


#### 2.2.1. Dataset Size

**Goal:** Quantify the scale of each cell's data to make cost-aware engineering decisions in subsequent analysis.

We will:
1. Run **dry runs across all 8 cells** to estimate their BigQuery scan size ($0 cost)
2. Compare estimates with known sizes from `TECHNICAL_SPECIFICATION.md`
3. Calculate real query costs and establish safe engineering patterns

> **Why this matters:** A careless `SELECT *` on a full cell could scan **~1 TiB** and cost **~$6.25** per query. With free tier at 1 TiB/month, we can afford only ~1 full scan before being billed. Understanding size = engineering responsibly.

In [18]:
# DATASET SIZE & COST ANALYSIS (DRY RUN, $0 cost)
print("=" * 70)
print("DATASET SIZE & COST ANALYSIS")
print("All estimates via DRY RUN \u2014 No Data Scanned, $0 Cost")
print("=" * 70)

# Known uncompressed sizes from TECHNICAL_SPECIFICATION.md \u00a72.2
known_sizes_gb = {
    'a': 1016, 'b': 941, 'c': 1077, 'd': 1027,
    'e': 819,  'f': 1035, 'g': 879, 'h': 928
}

print(f"\n{'Cell':<6} {'Known Size':<14} {'DryRun Est.':<14} {'Est. TiB':<12} {'Cost @ $6.25':<16} {'Note'}")
print("-" * 78)

rows = []
for cell in list('abcdefgh'):
    # Dry run: SELECT * on the largest table to estimate full scan cost
    sql = f"""
    SELECT *
    FROM `google.com:google-cluster-data.clusterdata_2019_{cell}.instance_usage`
    LIMIT 1
    """
    try:
        job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True))
        est_bytes = job.total_bytes_processed
        est_gb = est_bytes / 1e9
        est_tib = est_bytes / (1024**4)
        est_cost = est_tib * 6.25
        
        note = "\U0001f7e2 Smallest" if cell == 'e' else ("\U0001f534 Largest" if cell == 'c' else "")
        
        print(f"{cell:<6} {known_sizes_gb[cell]:>5} GB (GCS)  {est_gb:>8.0f} GB  {est_tib:>8.5f}   ${est_cost:<8.2f}  {note}")
        
        # Track estimate (dry run = no real cost, but useful for awareness)
        update_cost_tracker(est_bytes)
        rows.append({'cell': cell, 'est_gb': est_gb, 'est_tib': est_tib, 'est_cost': est_cost})
        
    except Exception as e:
        print(f"{cell:<6} ERROR: {e}")

print("\n" + "=" * 70)
print("COST IMPLICATIONS")
print("=" * 70)

# Use the smallest cell for cost analysis
cell_e = known_sizes_gb['e']
cell_e_tib = cell_e / 1024
cell_c_tib = known_sizes_gb['c'] / 1024
total_tib = sum(known_sizes_gb.values()) / 1024

print(f"\n\U0001f4ca Cell 'e' (smallest): {cell_e} GB \u2248 {cell_e_tib:.2f} TiB")
print(f"   \u2022 Full scan cost: \u2022 ${cell_e_tib * 6.25:.2f}")
print(f"   \u2022 1-day slice (\u00f731): ${cell_e_tib * 6.25 / 31:.2f}")
print(f"   \u2022 Free tier budget (1 TiB): ~{int(1 / cell_e_tib * 31)} day-filtered queries")

print(f"\n\U0001f4ca Cell 'c' (largest): {known_sizes_gb['c']} GB \u2248 {cell_c_tib:.2f} TiB")
print(f"   \u2022 Full scan cost: ${cell_c_tib * 6.25:.2f}")

print(f"\n\U0001f4ca All 8 cells combined: {sum(known_sizes_gb.values()):,} GB \u2248 {total_tib:.1f} TiB")
print(f"   \u2022 Cost to scan everything once: ${total_tib * 6.25:.2f}")
print(f"   \u2022 That's {total_tib:.0f}x the free tier!")

DATASET SIZE & COST ANALYSIS
All estimates via DRY RUN — No Data Scanned, $0 Cost

Cell   Known Size     DryRun Est.    Est. TiB     Cost @ $6.25     Note
------------------------------------------------------------------------------
a       1016 GB (GCS)      2260 GB   2.05539   $12.85     
Estimated scan: 2259.921 GB
Running total (estimated): 2.197054 TiB / 1.0 TiB free
b        941 GB (GCS)      2153 GB   1.95784   $12.24     
Estimated scan: 2152.667 GB
Running total (estimated): 4.154893 TiB / 1.0 TiB free
c       1077 GB (GCS)      2450 GB   2.22796   $13.92     🔴 Largest
Estimated scan: 2449.671 GB
Running total (estimated): 6.382856 TiB / 1.0 TiB free
d       1027 GB (GCS)      2351 GB   2.13816   $13.36     
Estimated scan: 2350.928 GB
Running total (estimated): 8.521013 TiB / 1.0 TiB free
e        819 GB (GCS)      1925 GB   1.75084   $10.94     🟢 Smallest
Estimated scan: 1925.071 GB
Running total (estimated): 10.271854 TiB / 1.0 TiB free
f       1035 GB (GCS)      2297 GB  

**Engineering Rules:**
1. NEVER SELECT * FROM full_table without a WHERE filter
2. ALWAYS add WHERE start_time BETWEEN ... for time-bounded queries
3. ALWAYS run dry_run=True before executing new queries
4. ALWAYS select only the columns you need (columnar = pay per column)
5. START with cell 'e' and 1 day of data (~$0.16/query)

## 3. Schema Information

**Goal:** Understand table schemas without spending money.

**Approach:**
1. Try `INFORMATION_SCHEMA.TABLES` (free, but blocked for public datasets → **expected** 403)
2. Fallback: `SELECT * ... LIMIT 0` (free, returns schema, 0 bytes scanned)
3. Official dataset documentation as authoritative reference


### 3.1. Schema via client.get_table() ($0 cost)

`client.get_table()` calls the BigQuery metadata REST API — no query, no data scanned, **completely free**.

**Note:** `get_table()` returns the full schema including nested RECORD fields (e.g. `average_usage.cpus`, `capacity.memory`). This is the **documented** way to get table metadata — unlike `INFORMATION_SCHEMA`, it works for public datasets.

In [20]:
# FREE schema retrieval via client.get_table() (metadata API, $0 cost)
print("=== TABLE SCHEMAS (via client.get_table()) ===\n")

from google.cloud.bigquery import DatasetReference, TableReference

dataset_ref = DatasetReference(
    "google.com:google-cluster-data", "clusterdata_2019_e"
)

tables = [
    'machine_events',
    'machine_attributes',
    'collection_events',
    'instance_events',
    'instance_usage',
]

for tbl in tables:
    try:
        table_ref = TableReference(dataset_ref, tbl)
        table = client.get_table(table_ref)
        print(f"\n--- {tbl} ({len(table.schema)} columns) ---")
        for field in table.schema:
            nested = f" ({', '.join(f.name for f in field.fields)})" if field.fields else ""
            print(f"  {field.name:<28} {field.field_type:<10} {field.mode:<6}{nested}")
    except Exception as e:
        print(f"  \u274c {tbl}: {e}")

print("\n\u2705 All schemas retrieved (metadata API).")


=== TABLE SCHEMAS (via client.get_table()) ===


--- machine_events (7 columns) ---
  time                         INTEGER    NULLABLE
  machine_id                   INTEGER    NULLABLE
  type                         INTEGER    NULLABLE
  switch_id                    STRING     NULLABLE
  capacity                     RECORD     NULLABLE (cpus, memory)
  platform_id                  STRING     NULLABLE
  missing_data_reason          INTEGER    NULLABLE

--- machine_attributes (5 columns) ---
  time                         INTEGER    NULLABLE
  machine_id                   INTEGER    NULLABLE
  name                         STRING     NULLABLE
  value                        STRING     NULLABLE
  deleted                      BOOLEAN    NULLABLE

--- collection_events (17 columns) ---
  time                         INTEGER    NULLABLE
  type                         INTEGER    NULLABLE
  collection_id                INTEGER    NULLABLE
  scheduling_class             INTEGER    NULLABLE
  mis

## 4. Conclusion

### 4.1. Accomplishments

This notebook successfully profiled the **Google ClusterData 2019** dataset at **$0 cost**. Here is what we achieved:

| # | Objective | Outcome |
|---|-----------|---------|
| 1 | **BigQuery Connection** | ✅ Verified and authenticated to `google.com:google-cluster-data` via project `sembada-cloud-research-2026` |
| 2 | **Cost Awareness** | ✅ Dry run on cell `e` (`instance_usage`, 3 columns, `LIMIT 5`) estimates **155 GB** \u2014 proving `LIMIT` does NOT reduce bytes scanned |
| 3 | **Per-Cell Size Analysis** | ✅ All 8 cells quantified. Cell `e` is smallest at **~819 GB** (dry-run estimate: ~1,925 GB). Cell `c` largest at **~1,077 GB** (~2,450 GB dry run). Full dataset ~7.5 TiB total |
| 4 | **Cost Implications** | ✅ A full `SELECT *` scan costs **$10\u201314/cell**. A 1-day filtered slice costs **~$0.16**. Free tier (1 TiB/month) supports ~38 targeted queries |
| 5 | **INFORMATION_SCHEMA** | ⚠️ Blocked (403 expected) \u2014 public datasets restrict project-scoped metadata |
| 6 | **Table Schemas** | ✅ Retrieved all 5 table schemas via `client.get_table()` ($0 metadata API). 5\u201318 columns each, with RECORD and REPEATED types documented |
| 7 | **Engineering Rules** | ✅ Established: always use `WHERE` filters, select only needed columns, run `dry_run=True` first, start with cell `e` and 1-day windows |

### 4.2. Schema Summary

| Table | Columns | Key Fields for ML |
|-------|---------|-------------------|
| `machine_events` | 7 | `capacity.cpus`, `capacity.memory`, `platform_id` \u2014 machine capacity bounds |
| `machine_attributes` | 5 | `name`, `value` \u2014 sparse key-value hardware metadata |
| `collection_events` | 17 | `priority`, `scheduling_class`, `user` \u2014 job context and preemption risk |
| `instance_events` | 13 | `resource_request.cpus`, `resource_request.memory`, `machine_id` \u2014 requested vs. actual usage |
| `instance_usage` | 18 | `average_usage`, `maximum_usage`, `cpu_usage_distribution` (11 pt), `tail_cpu_usage_distribution` (9 pt) \u2014 **primary ML target** |

### 4.3. Key Insight: Dry-Run Estimates vs. Actual Sizes

The dry-run estimates (~1,900\u20132,400 GB per cell for `SELECT *` on `instance_usage` alone) are approximately **2\u20132.5\u00d7 larger** than the known GCS compressed sizes (~819\u20131,077 GB per cell for all 5 tables). This is because:
- GCS sizes are **compressed JSON** across all 5 tables
- BigQuery dry runs report **decompressed columnar read size** for the selected columns only
- The `cpu_usage_distribution` and `tail_cpu_usage_distribution` arrays (20 values per row) inflate the logical scan size

### 4.4. Engineering Rules (Must Follow in EDA)

1. ❌ **Never** `SELECT *` from a full table without a `WHERE` filter
2. ✅ **Always** add `WHERE start_time BETWEEN ...` for time-bounded queries
3. ✅ **Always** run `dry_run=True` before executing new queries
4. ✅ **Always** select only the columns you need (columnar storage = pay per column)
5. ✅ **Start** with cell `e` and a 1-day window (~$0.16/query)

### 4.5. Cost Summary

| Phase | Actual Cost | Method |
|-------|-------------|--------|
| Data Profiling (this notebook) | **$0.00** | Dry runs, `client.get_table()` metadata API, constant expression queries |
| EDA Phase 1 (planned) | **~$0.50\u20132.00** | Targeted column-pruned, time-filtered queries on cell `e` |

### 4.6. Next Steps

1. **Extract 1 day of data** from cell `e` (May 1, 2019) using `WHERE start_time BETWEEN` with specific column selection
2. **Analyze CPU/memory utilization** \u2014 compare `resource_request` vs. `average_usage` to detect over-provisioning
3. **Handle nested types** \u2014 flatten RECORD fields and UNNEST REPEATED arrays for tabular analysis
4. **Reference authoritative schemas** from \u00a73.1 and the [Google cluster-usage traces v3](https://drive.google.com/file/d/10r6cnJ5cJ89fPWCgj7j4LtLBqYN9RiI9/view) specification
